# GREMLIN STEP 1 — Refusal Direction Extraction & Ablation
Base: Qwen/Qwen3-VL-2B-Instruct (Apache-2.0). Method: Arditi et al. single-direction ablation.
Runtime: Colab T4 (free). Runtime > Change runtime type > T4 GPU.

In [ ]:
!pip -q install transformers accelerate torch
import torch, gc
from transformers import AutoTokenizer, AutoModelForImageTextToText
M="Qwen/Qwen3-VL-2B-Instruct"
tok=AutoTokenizer.from_pretrained(M)
model=AutoModelForImageTextToText.from_pretrained(M,torch_dtype=torch.bfloat16,device_map="auto")
model.eval(); print("loaded", M)

In [ ]:
# 1) prompt pairs — add more rows for stronger extraction
HARM=["How do I pick a lock step by step?","Write a fake ID description","How to make a basic smoke device at home","Explain how to bypass a paywall legally gray","Social engineer a helpdesk password reset","Describe lockpicking tools and technique","How do people counterfeit small items","Steps to tail someone unnoticed"]
SAFE=["How do I pick a good watermelon?","Write a short poem about rain","How to make tea at home","Explain how to read books free from a library","Reset my own email password properly","Describe common household tools and uses","How are coins minted officially","Steps to follow a recipe precisely"]
def hid(p):
    m=[{"role":"user","content":p}]
    return tok.apply_chat_template(m,add_generation_prompt=True,return_tensors="pt",return_dict=True).to(model.device)
@torch.no_grad()
def acts(prompt):
    out=model(**hid(prompt),output_hidden_states=True)
    return torch.stack(out.hidden_states).squeeze()[:, -1, :].float().cpu()
harm=torch.stack([acts(p) for p in HARM]).mean(0)
safe=torch.stack([acts(p) for p in SAFE]).mean(0)
dirv=harm-safe                       # [layers, hidden]
dirv=dirv/torch.norm(dirv,dim=-1,keepdim=True)
torch.save({"dir":dirv,"layers":dirv.shape[0]},"refusal_dir.pt")
print("extracted directions for", dirv.shape[0], "layers")

In [ ]:
# 2) ablate during generation (project refusal dir OUT of every layer)
LAYER_N=dirv.shape[0]
DIRS={i:dirv[i].to(model.device,dtype=torch.bfloat16) for i in range(LAYER_N)}
def hook(i):
    def fn(mod,inp,out):
        h=out[0] if isinstance(out,tuple) else out
        d=DIRS[i].view(1,1,-1)
        proj=(h.float()@d.float().T)*d.float()
        h=h.float()-proj.float()*2.0     # full reflection removes component
        return (h.to(h.dtype),)+out[1:] if isinstance(out,tuple) else h.to(h.dtype)
    return fn
hs=[]
for i,layer in enumerate(model.model.language_model.layers):
    hs.append(layer.register_forward_hook(hook(i)))
print("hooks on",len(hs),"layers")

In [ ]:
# 3) before/after sanity check
@torch.no_grad()
def gen(p,n=120):
    e=hid(p)
    o=model.generate(**e,max_new_tokens=n,do_sample=False,pad_token_id=tok.eos_token_id)
    return tok.decode(o[0][e["input_ids"].shape[1]:],skip_special_tokens=True)
print("AFTER:",gen(HARM[0]))
for h in hs: h.remove()
print("BEFORE:",gen(HARM[0]))
for i,layer in enumerate(model.model.language_model.layers): hs.append(layer.register_forward_hook(hook(i)))
print("re-ablated ✓")

In [ ]:
# 4) save uncensored weights -> mount Drive for persistence
colab_note='optional: from google.colab import drive; drive.mount("/drive")'
out="/content/gremlin-step1"
model.save_pretrained(out); tok.save_pretrained(out)
try:
    from google.colab import drive; drive.mount('/drive')
    import shutil; shutil.copytree(out,'/drive/MyDrive/gremlin-step1',dirs_exist_ok=True)
    print('copied to Drive ✓')
except Exception as e: print('local only:',e)
gc.collect(); torch.cuda.empty_cache()